## Data Cleaning Notebook

This is where we ingest files from `raw_data/`, process them appropriately, and store the processed versions in `processed_data/`.

### Modules to Import (featured in `requirements.txt`)

In [100]:
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import contextily as cx

### Notebook Constants

In [101]:
RAW_DATA_PATH = "../raw_data/"        # read files
PROC_DATA_PATH = "../processed_data/" # write files
CRS = 3310                            # "California Albers" projection mapping, since we're using CA data

PARCEL_COLS = ["APN", "UseCode", "Land", "Imps", "LatestDocu", "SitusCity"] # columns from parcels.geojson we plan to keep

# These have to do with parcel "use codes" (i.e. what sector is it based in).
# We refer to this: https://propinfo.acgov.org/UseCodeList/ in order to figure out what to drop.
# A use code is 4 digits.
DROP_SERIES   = {"4", "6"}            # DROP IF THIS IS FIRST DIGIT. industrial, institutional
DROP_PREFIXES = {"73", "74", "75"}    # "..." FIRST TWO DIGITS. ondos, co-ops, deed-restricted affordable
DROP_CODES    = {"1130", "1150", "1160", "1600", "3900", "9401", "9405", "9910"} # "..." ALL 4 DIGITS

# "DU" means "dwelling units"
# depending on the tier/category there is more DU.
DU_PER_ACRE = {
    "Tier 1 - 200ft": 160, "Tier 1 - Quarter Mile": 120, "Tier 1 - Half Mile": 100,
    "Tier 2 - 200ft": 140, "Tier 2 - Quarter Mile": 100, "Tier 2 - Half Mile": 80,
}

### Ingesting / Inspecting Raw Data

In [ ]:
epc_data = gpd.read_file(RAW_DATA_PATH + "epc_data.geojson")
tod_data = gpd.read_file(RAW_DATA_PATH + "tod_zones.geojson")

tod_mask = tod_data.to_crs(3857).union_all() # we want parcels in tod zones, 
                                             # so we would like to match projections (3857: spherical mercator)

parcels_data = gpd.read_file(RAW_DATA_PATH + "parcels.geojson", # takes a while to load... (> 1 GB!!)
                             engine="pyogrio", use_arrow=True,
                             columns=PARCEL_COLS, mask=tod_mask)

parcels_data = (parcels_data
                .set_crs(3857, allow_override=True)      # data is assumed be in CRS 4326, but actually 3857. so label accordingly first
                .to_crs(CRS))                            # then reproject to 3310


In [ ]:
parcels_data.to_parquet(PROC_DATA_PATH + "parcels_tod.parquet") # .parquet is faster to load

In [ ]:
print("EPC dims:", epc_data.shape)
print("Parcel dims:", parcels_data.shape)
print("TOD zone dims:", tod_data.shape)

In [ ]:
epc_data.columns

In [ ]:
parcels_data.columns

In [ ]:
tod_data.columns

### Filtering and Joining

Main things to look out for: 
- valid "use codes" for parcels
- tracking parcels that lie in TOD (Transit-Oriented Development) Zones
- tracking TOD parcels that are in an EPC (Equity Priority Community) Area

In [ ]:
epc_data = epc_data.to_crs(CRS)
tod_data = tod_data.to_crs(CRS)

In [ ]:
parcels_data['geometry'].isna().any()                                                     # are there null geometries?
parcels_data = parcels_data.drop_duplicates(subset=['geometry'])                          # avoid double-counting of parcels
parcels_data = parcels_data[~((parcels_data['Land'] == 0) & (parcels_data['Imps'] == 0))] # remove exempt/public land
parcels_data.shape

In [ ]:
code = parcels_data["UseCode"]
drop = (
    code.isna()
    | code.str[0].isin(DROP_SERIES)
    | code.str[:2].isin(DROP_PREFIXES)
    | code.isin(DROP_CODES)
)

parcels_data = parcels_data[~drop]
print(parcels_data.shape)

In [ ]:
epc_data = epc_data[(epc_data['county_fip'] == 1) | (epc_data['county_fip'] == 85)] # keep only Alameda (1), Santa Clara (85)
epc_data.shape

In [ ]:
tod_data.zone_label.unique()

In [ ]:
# SPATIAL JOIN 1: PARCELS + TOD ZONES

tod_data["du_per_acre"] = tod_data["zone_label"].map(DU_PER_ACRE)
parcels_data["acres"] = parcels_data.area / 4046.86

pts = parcels_data.copy()
pts.geometry = parcels_data.representative_point()
pts = gpd.sjoin(pts, tod_data[["zone_id", "zone_label", "du_per_acre", "geometry"]],
                predicate="within") # SPATIAL JOIN WITH TOD ZONES

# keep each parcel's most permissive zone
pts = pts.sort_values("du_per_acre", ascending=False)
pts = pts[~pts.index.duplicated()]

pts["max_units"] = np.floor(pts["acres"] * pts["du_per_acre"]).astype(int)

# This is where we track parcels in TOD zones (used pts as a temp variable)
parcels_tod = parcels_data.join(pts[["zone_id", "zone_label", "du_per_acre", "max_units"]],
                                how="inner")

In [ ]:
parcels_tod = parcels_tod[parcels_tod["max_units"] >= 5].copy() # SB 79 projects need at least 5 units

In [ ]:
# SPATIAL JOIN 2: PARCELS / TOD ZONES + EPC DATA

reps = parcels_tod.copy()
reps.geometry = parcels_tod.representative_point()

reps = gpd.sjoin(reps, epc_data[["epc_class", "geometry"]], how="left", predicate="within")
reps = reps[~reps.index.duplicated()]

parcels_tod["epc_class"] = reps["epc_class"]
parcels_tod["is_epc"] = parcels_tod["epc_class"].notna()

In [ ]:
EPC_ORDER = ["Not EPC", "High", "Higher", "Highest"]
parcels_tod["epc_class"] = pd.Categorical(parcels_tod["epc_class"].fillna("Not EPC"),
                                          categories=EPC_ORDER, ordered=True)

# numbers stored as text should be converted to actual numbers
for col in ["Land", "Imps", "LatestDocu"]:
    parcels_tod[col] = pd.to_numeric(parcels_tod[col], errors="coerce")

# soft sites are "buildings worth less than half the land"
parcels_tod["il_ratio"] = parcels_tod["Imps"] / parcels_tod["Land"].replace(0, np.nan)
parcels_tod["soft_site"] = parcels_tod["il_ratio"] < 0.5
parcels_tod["soft_units"] = parcels_tod["max_units"].where(parcels_tod["soft_site"], 0)

# recent activity: a recorded sale or document in the last 10 years
parcels_tod["recent_activity"] = parcels_tod["LatestDocu"] >= 2016

### Finalize our Dataset!

In [ ]:
parcels_tod.shape # able to reduce our parcel set by ~40% of rows

In [ ]:
parcels_tod.columns # identify what attributes we care about now.

In [ ]:
parcels_tod.head()

In [ ]:
print("total allowed units:", parcels_tod["max_units"].sum())
print("share in an epc:", parcels_tod["is_epc"].mean().round(3)) # this number is ~50%! 
print(parcels_tod["zone_label"].value_counts())

### Visualizing our Processed Dataset

This map scopes to Alameda County only. We plan to scope to Santa Clara County in our final project submission.

In [ ]:
parcels_map = parcels_tod.to_crs(3857)
epc_map = epc_data.to_crs(3857)

fig, ax = plt.subplots(figsize=(10, 10))
parcels_map.plot(ax=ax, column="du_per_acre", categorical=True, cmap="YlOrRd",
                 linewidth=0, legend=True,
                 legend_kwds={"title": "SB 79 min. density (du/acre)", "loc": "upper right"})
epc_map.boundary.plot(ax=ax, color="black", linewidth=0.5)

xmin, ymin, xmax, ymax = parcels_map.total_bounds
ax.set_xlim(xmin - 2000, xmax + 2000)
ax.set_ylim(ymin - 2000, ymax + 2000)

epc_map.plot(ax=ax, color="none", edgecolor="black", linewidth=0.5, hatch="///", alpha=0.4)  # hatched = epc
tod_data.to_crs(3857).boundary.plot(ax=ax, color="gray", linewidth=0.3)                        # zone rings
cx.add_basemap(ax, source=cx.providers.Esri.WorldGrayCanvas) 
ax.set_axis_off()
ax.set_title("SB 79 allowed density, Alameda County TOD parcels\n(hatched areas = Equity Priority Communities)")
plt.show()

### Save to `processed_data`
We use `.parquet` for efficient loading.

In [ ]:
parcels_tod.to_parquet(PROC_DATA_PATH + "parcels_tod_clean.parquet")
print('loaded cleaned data')